In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute the function values
x = np.linspace(-40 * np.pi/180, 40 * np.pi/180, 1000)
y = np.exp(-10 * np.abs(x))

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x, y, color='darkblue', linewidth=2)

# Add decorations
plt.title(r'Eating Probability vs. Orientation to Food', fontsize=16)
plt.xlabel(r'$\mathrm{\theta_{eating}}$', fontsize=14)
plt.ylabel('Eating Probability', fontsize=14)
plt.grid(alpha=0.3)

# Relabel x-axis in degrees
x_ticks = np.linspace(-40 * np.pi/180, 40 * np.pi/180, 5)  # Define tick positions in radians
x_labels = [f"{np.degrees(tick):.0f}°" for tick in x_ticks]  # Convert to degrees
plt.xticks(ticks=x_ticks, labels=x_labels)

plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import seaborn as sns

# Set Seaborn style
sns.set(style="whitegrid")

# Plot the function with Seaborn styling
plt.figure(figsize=(10, 6))
plt.plot(x, y, color=sns.color_palette("muted")[0], linewidth=2)

# Add decorations
# plt.title(r'Eating Probability vs. Orientation to Food', fontsize=16)
# plt.xlabel(r'$\mathrm{\theta_{eating}}$', fontsize=20)
# plt.ylabel('Eating Probability', fontsize=20)
plt.grid(True, alpha=0.3, linestyle='--')

# Relabel x-axis in degrees
plt.xticks(ticks=x_ticks, labels=x_labels, fontsize=30)
plt.yticks(fontsize=30)

# Adjust x-axis limits to make the plot more condensed
plt.xlim(x_ticks[0], x_ticks[-1])
plt.ylim(-0.0, 1.0)

plt.tight_layout()

# Show the plot
plt.savefig("eating_probability_vs_orientation.svg", format='svg')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a grid of possible actions
move_actions = np.linspace(0, 1, 100)
turn_actions = np.linspace(-1, 1, 100)
M, T = np.meshgrid(move_actions, turn_actions)

# Apply the transformation
T_transformed = T * (1 - M)

# Plot the action space
plt.figure(figsize=(8, 6))
plt.scatter(M, T_transformed, c=T_transformed, cmap='coolwarm', s=5)
plt.xlabel("move_action")
plt.ylabel("turn_action (transformed)")
plt.title("Action Space after Transformation: turn_action * (1 - move_action)")
plt.colorbar(label="Turn Action Value")
plt.grid(True)
plt.show()


In [ ]:
import cfg

# Define the vertices of the triangle
triangle_vertices = [(0, 1), (0, -1), (1, 0)]

# Extract x and y coordinates of the vertices
triangle_x, triangle_y = zip(*triangle_vertices)

# Close the triangle by appending the first vertex at the end
triangle_x += (triangle_x[0],)
triangle_y += (triangle_y[0],)

# Plot the triangle
plt.figure(figsize=(6, 6))
plt.plot(triangle_x, triangle_y, color='black', linewidth=2)
plt.fill(triangle_x, triangle_y, color='lightblue', alpha=0.5)

# Set axis limits
plt.xlim(-0.005, 1)
plt.ylim(-1, 1)

# Relabel x and y axes by scaling them
x_ticks = np.linspace(0, 1, 3)  # Define tick positions in the range [0, 1]
x_ticks_scaled = [tick * cfg.FISH_CONSTANTS["max_speed"] for tick in x_ticks]
y_ticks = np.linspace(-1, 1, 5)  # Define fewer tick positions
y_ticks_scaled = [tick * cfg.FISH_CONSTANTS["max_turn_speed"] for tick in y_ticks]
plt.yticks(ticks=y_ticks, labels=[f"{tick:.1f}" for tick in y_ticks_scaled], fontsize=40)

plt.xticks(ticks=x_ticks, labels=[f"{tick:.2f}" for tick in x_ticks_scaled], fontsize=40)
plt.yticks(ticks=plt.yticks()[0], labels=[f"{tick:.2f}" for tick in y_ticks_scaled], fontsize=40)

# Add labels and title
#plt.xlabel('Forward Speed (mm/s)', fontsize=30)
#plt.ylabel('Turn Speed (rad/s)', fontsize=30)

# Show grid and plot
plt.tight_layout()
plt.grid(True, alpha=0.3, linestyle='--')
plt.savefig("action_space_triangle.svg", format='svg')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sample 100 points within the action space
num_samples = 100000
action_noise = 0.1

# create a 2D grid of actions (approx. num_samples points)
n_side = int(np.sqrt(num_samples))
x_lin = np.linspace(0, 1, n_side)
y_lin = np.linspace(-1, 1, n_side)
X, Y = np.meshgrid(x_lin, y_lin)
x_samples_raw = X.ravel()
y_samples_raw = Y.ravel()

# update num_samples to the actual number of grid points (optional)
num_samples = x_samples_raw.size

sampled_points = list(zip(x_samples_raw, y_samples_raw))

print(sampled_points)
# Initialize lists to store all trajectories
all_trajectories = []


# Compute each trajectory
for move_forward, turn in sampled_points:
    # Define the starting location for the current trajectory

    penalize_turn_threshold = 0.3
    penalize_move_threshold = 0.3

    energy_cost = 0
    if move_forward > penalize_move_threshold:
        energy_cost += (move_forward - penalize_move_threshold)

    if abs(turn) > penalize_turn_threshold:
        energy_cost += (abs(turn) - penalize_turn_threshold)

    # Apply the transformation to y_samples
    turn = turn * (1 - move_forward)
    move_forward = move_forward * 5
    turn = turn * 7

    # Scale x and y by 0.2
    move_forward = move_forward / 8
    turn = turn / 8

    start_x, start_y = 0, 0
    trajectory_x = [start_x]
    trajectory_y = [start_y]
    
    # Update the direction based on the turn value
    direction = turn

    move_forward = move_forward * (1 + np.random.uniform(-action_noise*np.sqrt(12)/2, action_noise*np.sqrt(12)/2))
    turn = turn * (1 + np.random.uniform(-action_noise*np.sqrt(12)/2, action_noise*np.sqrt(12)/2))

    # Compute the new position
    new_x = trajectory_x[-1] + move_forward * np.cos(turn)
    new_y = trajectory_y[-1] + move_forward * np.sin(turn)
    
    # Append the new position to the trajectory
    trajectory_x.append(new_x)
    trajectory_y.append(new_y)
    
    # Store the trajectory
    all_trajectories.append(((trajectory_x, trajectory_y), energy_cost))

# Plot all trajectories
# plt.figure(figsize=(10, 6))
# for trajectory_x, trajectory_y in all_trajectories:
#     plt.plot(trajectory_x, trajectory_y, marker='o', markersize=2, linestyle='-', alpha=0.7)

# plt.title("Trajectories Plot", fontsize=16)
# plt.xlabel("X Position", fontsize=14)
# plt.ylabel("Y Position", fontsize=14)
# plt.grid(alpha=0.3)
# plt.axis('equal')
# plt.show()


In [ ]:
# # Plot all trajectories
# plt.figure(figsize=(10, 6))
# for trajectory_x, trajectory_y in all_trajectories:
#     plt.plot(trajectory_x, trajectory_y, marker='o', markersize=2, linestyle='-', alpha=0.7)

# plt.title("Trajectories Plot", fontsize=16)
# plt.xlabel("X Position", fontsize=14)
# plt.ylabel("Y Position", fontsize=14)
# plt.grid(alpha=0.3)
# plt.axis('equal')
# plt.show()

# Extract end locations of trajectories
end_locations_x = [trajectory[0][-1] for trajectory, _ in all_trajectories]
end_locations_y = [trajectory[1][-1] for trajectory, _ in all_trajectories]

# Plot the distribution of end locations
# plt.figure(figsize=(10, 6))
# plt.scatter(end_locations_x, end_locations_y, alpha=0.7, c='blue', s=10)
# plt.title("Distribution of End Locations", fontsize=16)
# plt.xlabel("X Position", fontsize=14)
# plt.ylabel("Y Position", fontsize=14)
# plt.grid(alpha=0.3)
# plt.axis('equal')
# plt.show()


In [ ]:
from matplotlib.colors import LogNorm

# Create a figure with a larger size for better visibility
plt.figure(figsize=(12, 8))

# Plot the hexbin heatmap with improved aesthetics
hexbin_plot = plt.hexbin(
    end_locations_x, 
    end_locations_y, 
    gridsize=100, 
    cmap='viridis',  # Use a bluish colormap
    mincnt=1, 
    norm=LogNorm()
)

# Add a colorbar with a more descriptive label
cbar = plt.colorbar(hexbin_plot, label='Density (log scale)', pad=0.02)
cbar.ax.tick_params(labelsize=14)  # Increase colorbar tick label size

# Add title and axis labels with larger font sizes
plt.title("Heatmap of End Locations (Logarithmic Scale)", fontsize=20, pad=20)
plt.xlabel("X Position", fontsize=16, labelpad=10)
plt.ylabel("Y Position", fontsize=16, labelpad=10)

# Customize tick parameters for better readability
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Add a grid for better visual alignment
# plt.grid(alpha=0.3, linestyle='--')

# Ensure the aspect ratio is equal for accurate representation
plt.axis('equal')

# Save the plot with high resolution for publication
# plt.savefig("heatmap_end_locations_final.png", dpi=300, bbox_inches='tight')

# Show the plot
plt.tight_layout()
plt.savefig("heatmap_end_locations_final.svg", format='svg')
plt.show()

In [ ]:
from matplotlib.colors import Normalize

# Extract final positions and energies from all_trajectories
end_x = [traj[0][-1] for traj, _ in all_trajectories]
end_y = [traj[1][-1] for traj, _ in all_trajectories]
energies = np.array([energy for _, energy in all_trajectories])

# Robust normalization (handle constant energies)
vmin, vmax = energies.min(), energies.max()
if vmin == vmax:
    vmax = vmin + 1e-8
norm = Normalize(vmin=vmin, vmax=vmax)

# Scatter final positions colored by energy
plt.figure(figsize=(12, 8))
sc = plt.scatter(end_x, end_y, c=energies, cmap='viridis', norm=norm, s=20, edgecolors='k', linewidths=0.2, alpha=0.9)

cbar = plt.colorbar(sc, label='Energy cost', pad=0.02)
cbar.ax.tick_params(labelsize=12)

plt.title("Final Positions Colored by Energy Cost", fontsize=18)
plt.xlabel("X Position", fontsize=14)
plt.ylabel("Y Position", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.axis('equal')
plt.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig("final_positions_by_energy.svg", format='svg')
plt.show()


In [ ]:
from scipy.stats import gaussian_kde

# Prepare data
xy = np.vstack([end_locations_x, end_locations_y])
# Ensure energies is a numpy array
energies_arr = np.asarray(energies)

# KDE for density and KDE weighted by energy
# kde_density = gaussian_kde(xy)
kde_weighted = gaussian_kde(xy, weights=energies_arr)

# Grid for evaluation
x_min, x_max = min(end_locations_x), max(end_locations_x)
y_min, y_max = min(end_locations_y), max(end_locations_y)
x_grid = np.linspace(x_min, x_max, 500)
y_grid = np.linspace(y_min, y_max, 500)
X, Y = np.meshgrid(x_grid, y_grid)
positions = np.vstack([X.ravel(), Y.ravel()])

# Evaluate KDEs
# Z_density = kde_density(positions).reshape(X.shape)
Z_weighted = kde_weighted(positions).reshape(X.shape)

# # Compute local average energy (weighted / density), guard against division by zero
# eps = 1e-12
# Z_mean_energy = Z_weighted / (Z_density + eps)

# # Mask regions with negligible density to avoid showing spurious values
# density_threshold = Z_density.max() * 1e-4
# Z_mean_energy_masked = np.ma.masked_where(Z_density < density_threshold, Z_mean_energy)

# Plot
plt.figure(figsize=(12, 8))
cf = plt.contourf(X, Y, Z_weighted, levels=50, cmap='viridis')
cbar = plt.colorbar(cf, label='Mean Energy Cost')
cbar.ax.tick_params(labelsize=14)

plt.title("Smoothed Mean Energy Cost over End Locations", fontsize=20, pad=20)
plt.xlabel("X Position", fontsize=16, labelpad=10)
plt.ylabel("Y Position", fontsize=16, labelpad=10)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.axis('equal')
plt.tight_layout()
plt.savefig("smoothed_mean_energy_end_locations.svg", format='svg')
plt.show()


In [ ]:
# Recompute the reachability/energy map using your kinematics:
# 1) orientation <- orientation + theta_change
# 2) translate straight by s along the NEW orientation
#
# We'll keep the same action->real mapping you provided earlier so that
#   s = move_forward * (5/8) mm
#   theta_change = turn*(1 - move_forward) * (7/8) rad
#
# Initial orientation = 0 (facing +x).

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm


# Sampling
n_move = 1000
n_turn = 1000
move = np.linspace(0.0, 1.0, n_move)
turn = np.linspace(-1.0, 1.0, n_turn)
M, T = np.meshgrid(move, turn, indexing='xy')

# Energy thresholds in action space
penalize_move_threshold = 0.3
penalize_turn_threshold = 0.3

# Energy in action space
energy = np.zeros_like(M)
energy += np.clip((M - penalize_move_threshold) * 0.01, 0, None)
energy += np.clip((np.abs(T) - penalize_turn_threshold) * 0.01, 0, None)

# Action -> real mapping from user's scaling
s = M * (5.0/8.0)                 # max_step = 5/8 mm
theta = T * (1.0 - M) * (7.0/8.0) # max_turn = 7/8 rad

# Kinematics: rotate then straight-line translate
theta_final = theta  # initial orientation is zero
x = s * np.cos(theta_final)
y = s * np.sin(theta_final)

# Prepare for plotting
X = x.ravel()
Y = y.ravel()
E = energy.ravel()

plt.figure(figsize=(7, 4))
hb = plt.hexbin(Y, X, C=E, reduce_C_function=np.mean, gridsize=120)
ax = plt.gca()
# ax.set_axis_off()

# # Add a scale bar (0.1 mm) at bottom left
# scalebar_length = 0.1  # mm
# x0, y0 = -0.4, -0.45   # position for the bar (adjust as needed)
# ax.plot([x0, x0 + scalebar_length], [y0, y0], color='black', lw=2)
# ax.text(x0 + scalebar_length/2, y0 - 0.02, "0.1 mm",
#         ha='center', va='top', fontsize=10)

ax.set_aspect('equal', adjustable='box')
ax.set_axis_off()

# --- Add scale bar (0.1 mm) without changing hex layout ---
fontprops = fm.FontProperties(size=8)
scalebar = AnchoredSizeBar(ax.transData,     # interpret length in data coords
                           0.1,              # bar length in mm (data units)
                           '0.1 mm',         # label
                           loc='lower left', # anchor corner
                           pad=0.15,
                           color='black',
                           frameon=False,
                           size_vertical=0.006)  # bar thickness in data units
ax.add_artist(scalebar)

plt.xlabel("(mm)")  # y
plt.ylabel("(mm)")  # x
# plt.title("Energy cost to reach location after one bout (egocentric view)")
cb = plt.colorbar(hb)
cb.ax.set_title("Energy Cost", fontsize=12, pad=6)
plt.scatter([0],[0], marker='x')
plt.gca().set_aspect('equal', adjustable='box')
out_path_egocentric = "one_bout_energy_map.svg"
plt.tight_layout()
plt.savefig(out_path_egocentric, format='svg')
out_path_egocentric
